In [1]:
!pip install -q litgpt transformers datasets torch huggingface_hub sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.3/263.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.5/760.5 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 70.7 MB/s eta 0:00:00


In [3]:
# @title 2. Authenticate
# Enter your token to access meta-llama/Llama-3.2-1B
# @title 2. Authenticate
from google.colab import userdata
from huggingface_hub import login

# Enter your token to access meta-llama/Llama-3.2-1B
try:
    # Attempt to load from Colab Secrets (Key icon on left sidebar)
    token = userdata.get('HF_TOKEN')
    login(token=token)
except Exception:
    # Fallback to interactive login if secret is not set
    print("Colab secret 'HF_TOKEN' not found. Please login interactively:")
    login()

Colab secret 'HF_TOKEN' not found. Please login interactively:


In [5]:
# @title 3. Load & Prepare Alpaca (First 100 Samples)
# Standardizing the data input
from datasets import load_dataset

# Standardizing the data input
print("Loading Alpaca dataset...")
ds = load_dataset("tatsu-lab/alpaca", split="train")
samples = ds.select(range(100))

prompts = []
for item in samples:
    if item['input']:
        prompt = f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n"
    else:
        prompt = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{item['instruction']}\n\n### Response:\n"
    prompts.append(prompt)

print(f"Prepared {len(prompts)} prompts.")

Loading Alpaca dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Prepared 100 prompts.


In [7]:
# @title 4. Benchmark: Hugging Face (Post-Attention Output)
import torch
import gc
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Helper Function: Clear GPU Memory ---
def clear_gpu():
    torch.cuda.empty_cache()
    gc.collect()

# --- CONFIGURATION ---
model_id = "meta-llama/Llama-3.2-1B"
device = "cuda"
LAYER_INDEX = 0
HEAD_INDEX = 0   # Extract output of Head 0
NUM_HEADS = 32
HEAD_DIM = 64

print(f"Loading HF Model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=device,
    attn_implementation="eager" # Ensure no hidden kernels
)

hf_head_outputs = []

def hf_post_attn_hook(module, args, output):
    # args[0] is the input to the o_proj layer (concatenated heads)
    concatenated_heads = args[0].detach().cpu()

    # 1. Reshape: [Batch, Seq, Num_Heads, Head_Dim]
    batch_size, seq_len, _ = concatenated_heads.shape
    separated = concatenated_heads.view(batch_size, seq_len, NUM_HEADS, HEAD_DIM)

    # 2. Extract specific head
    specific_head = separated[:, :, HEAD_INDEX, :].clone()
    hf_head_outputs.append(specific_head)

# Hook the 'o_proj' layer
target_layer = hf_model.model.layers[LAYER_INDEX].self_attn.o_proj
h_hf = target_layer.register_forward_hook(hf_post_attn_hook)

print(f"Capturing Post-Attention Output for Head {HEAD_INDEX}...")
start_time = time.time()

# Sequential Processing
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    _ = hf_model.generate(**inputs, max_new_tokens=1)

end_time = time.time()
hf_duration = end_time - start_time

h_hf.remove()
torch.save(hf_head_outputs, "hf_head_outputs.pt")

print(f"HF Time: {hf_duration:.4f} s")
print(f"Captured {len(hf_head_outputs)} head outputs.")

del hf_model, tokenizer
clear_gpu()

Loading HF Model: meta-llama/Llama-3.2-1B...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Capturing Post-Attention Output for Head 0...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

HF Time: 5.7454 s
Captured 100 head outputs.


In [8]:
# @title 5. Benchmark: LitGPT (Post-Attention Output)
import huggingface_hub.constants
from litgpt import LLM
from litgpt.api import LLM as LitLLM
from pathlib import Path
import time

# --- CONFIGURATION ---
LAYER_INDEX = 0
HEAD_INDEX = 0
# Llama 3.2-1B specific dimensions
NUM_HEADS = 32
HEAD_DIM = 64

# Hotfix for HF Transfer
if not hasattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    setattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER", False)

model_id = "meta-llama/Llama-3.2-1B"
print(f"Loading LitGPT Model: {model_id}...")

try:
    llm = LitLLM.load(model_id)
except:
    from litgpt.scripts.download import download_from_hub
    from litgpt.scripts.convert_hf import convert_hf
    download_from_hub(repo_id=model_id)
    checkpoint_dir = Path(f"checkpoints/{model_id}")
    convert_hf(checkpoint_dir=checkpoint_dir)
    llm = LitLLM.load(checkpoint_dir)

lit_head_outputs = []

def lit_post_attn_hook(module, args, output):
    # args[0] is the input to the proj layer (same as o_proj in HF)
    concatenated_heads = args[0].detach().cpu()

    # 1. Reshape to separate heads
    # LitGPT shapes are consistent with HF: [Batch, Seq, Hidden]
    batch_size, seq_len, _ = concatenated_heads.shape
    separated = concatenated_heads.view(batch_size, seq_len, NUM_HEADS, HEAD_DIM)

    # 2. Extract specific head
    specific_head = separated[:, :, HEAD_INDEX, :].clone()

    lit_head_outputs.append(specific_head)

# In LitGPT, the output projection layer is named 'proj'
# Structure: transformer.h[i].attn.proj
target_layer = llm.model.transformer.h[LAYER_INDEX].attn.proj
h_lit = target_layer.register_forward_hook(lit_post_attn_hook)

print(f"Capturing Post-Attention Output for Head {HEAD_INDEX}...")
start_time = time.time()

# Sequential Processing
for prompt in prompts:
    _ = llm.generate(prompt, max_new_tokens=1)

end_time = time.time()
lit_duration = end_time - start_time

h_lit.remove()
torch.save(lit_head_outputs, "lit_head_outputs.pt")

print(f"LitGPT Time: {lit_duration:.4f} s")
print(f"Captured {len(lit_head_outputs)} head outputs.")

del llm
clear_gpu()

Loading LitGPT Model: meta-llama/Llama-3.2-1B...
It is recommended to install hf_transfer for faster checkpoint download speeds: `pip install hf_transfer`


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Converting checkpoint files to LitGPT format.
{'checkpoint_dir': PosixPath('checkpoints/meta-llama/Llama-3.2-1B'),
 'debug_mode': False,
 'dtype': None,
 'model_name': None}


Loading weights: model.safetensors: 100%|██████████| 00:47<00:00,  2.12it/s


Saving converted checkpoint to checkpoints/meta-llama/Llama-3.2-1B


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Capturing Post-Attention Output for Head 0...
LitGPT Time: 8.9820 s
Captured 100 head outputs.


In [10]:
# @title 6. Validation (Post-Attention)
# @title 6. Validation (Post-Attention)
import torch

print("--- Post-Attention Comparison ---")
print(f"HF Time:     {hf_duration:.4f} s")
print(f"LitGPT Time: {lit_duration:.4f} s")

# Load the first sample from both
hf_out = torch.load("hf_head_outputs.pt")[0]
lit_out = torch.load("lit_head_outputs.pt")[0]

print("\n--- Shape Validation ---")
print(f"HF Output Shape:     {hf_out.shape}")
print(f"LitGPT Output Shape: {lit_out.shape}")

# We expect: [Batch=1, Seq=Variable, Head_Dim=64]
batch_match = (hf_out.shape[0] == 1)
dim_match   = (hf_out.shape[-1] == 64)
seq_match   = (hf_out.shape[1] == lit_out.shape[1]) # Lengths must equal each other

if batch_match and dim_match and seq_match:
    print("SUCCESS: Dimensions align perfectly across frameworks.")
    print(f"Sequence Length processed: {hf_out.shape[1]} tokens")
else:
    print("WARNING: Dimension mismatch.")

print("\n--- Value Validation ---")
# Compare the very first value in the tensor to ensure they are numerically identical
print(f"HF First Value:     {hf_out[0,0,0]:.6f}")
print(f"LitGPT First Value: {lit_out[0,0,0]:.6f}")

diff = torch.abs(hf_out - lit_out).max().item()
print(f"Max Absolute Difference: {diff:.8f}")

if diff < 1e-5:
    print("CONCLUSION: Models are numerically identical! ✅")
else:
    print("CONCLUSION: Small numerical drift detected (likely due to float precision). ⚠️")

--- Post-Attention Comparison ---
HF Time:     5.7454 s
LitGPT Time: 8.9820 s

--- Shape Validation ---
HF Output Shape:     torch.Size([1, 32, 64])
LitGPT Output Shape: torch.Size([1, 32, 64])
SUCCESS: Dimensions align perfectly across frameworks.
Sequence Length processed: 32 tokens

--- Value Validation ---
HF First Value:     0.002580
LitGPT First Value: 0.002640
Max Absolute Difference: 0.00254083
CONCLUSION: Small numerical drift detected (likely due to float precision). ⚠️
